#### RAGの構築方法　LlamaIndexライブラリ使用

LlamaIndex は、ドキュメント検索や自然言語処理のタスクを簡単に行うためのライブラリ。
- インデックス作成が簡単
- 検索機能が柔軟
- 言語モデル利用が前提で設計されている

LlamaIndex https://www.llamaindex.ai/

◆準備

In [9]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex
from llama_index.core import SimpleDirectoryReader
from llama_index.llms.openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")
os.environ['OPENAI_API_KEY']  = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

◆インデックスの構築

In [13]:
# ドキュメントからテキスト情報を読込
documents = SimpleDirectoryReader(input_dir='./data/text').load_data()

# インデックスの構築　（量が多いと止まることも；（トークン不足注意
index = VectorStoreIndex.from_documents(documents)
# パソコンのフォルダの中にあるファイルを読み込む、Data Connector。
# ほかにも、他にも多くの種類がある。　https://llamahub.ai/ 
# 読み込んだドキュメントを一定の長さの「チャンク」に分割し、ベクトル化まで一度にできる

インデックスには下記のような種類がある
Vector Store Index：埋め込みベクトルを使ってノードを抽出（基本）
List Index：単純な一覧形式
Tree Index：木構造を作成してたどる
Keyword Table Index：キーワードで抽出

In [ ]:
# （エラー確認作業用）カレントディレクトリの取得
import os 
print(os.getcwd())

c:\Users\user\Downloads\llmdev\14_rag


◆チャットエンジンの作成　問い合わせ用

In [14]:
# Chat Engineの作成
llm = OpenAI(model=MODEL_NAME)
chat_engine = index.as_chat_engine(
    chat_mode="openai", llm=llm, verbose=True
    )

.as_chat_engine()メソッド・・・チャットエンジンを利用。
mode は挙動を制御。best,openai(今回),react,condense_question　がある。
llm=llm（言語モデルオブジェクト指定）
verbose：True で詳細な出力を取得。


◆実験

In [15]:
# 質問：1回目
response = chat_engine.chat("有給休暇はいつから取得できますか？")

# 言語モデルからの回答を表示
print(response)

Added user message to memory: 有給休暇はいつから取得できますか？
=== Calling Function ===
Calling function: query_engine_tool with args: {"input":"有給休暇はいつから取得できますか？"}
Got output: 有給休暇は、入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に初めて付与されます。そのため、条件を満たした後に取得可能となります。

有給休暇は、入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に初めて付与されます。そのため、条件を満たした後に取得可能となります。


◆引用元を表示

In [16]:
# 引用元を表示
for source in response.sources:
    for source_node in source.raw_output.source_nodes:
        print("ファイル名：", source_node.metadata["file_name"])
        print("関連度スコア:", source_node.score)
        print("テキスト：")
        print(source_node.node.text)
        print("-" * 50)  # 区切り線

ファイル名： 03休暇規則.md
関連度スコア: 0.872725580018051
テキスト：
1. 年次有給休暇（有給休暇）

1. **有給休暇とは**

   - 有給休暇は、給与を受け取りながら休暇を取得できる制度です。
   - 心身のリフレッシュや私用のために自由に利用できます。

2. **付与日数**

   - 入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に、初めて有給休暇が付与されます。
   - 初年度は10日間の有給休暇が付与され、その後は勤続年数に応じて増加します。

     | 勤続年数 | 年次有給休暇日数 |
     |----------|----------------|
     | 0.5年    | 10日           |
     | 1.5年    | 11日           |
     | 2.5年    | 12日           |
     | 3.5年    | 14日           |
     | 4.5年    | 16日           |
     | 5.5年    | 18日           |
     | 6.5年以上 | 20日           |

3. **有給休暇の取得方法**

   - 有給休暇を取得する際は、原則として**3日前**までに上司に申請してください。
   - 緊急の場合は、当日の申請も可能ですが、できるだけ早めに連絡をお願いします。
   - 申請は、社内の休暇申請システムを利用してください。

4. **有給休暇の繰越し**

   - 未使用の有給休暇は、翌年度に限り繰り越すことができます。
   - 最大で40日間の有給休暇を保有することが可能です。
--------------------------------------------------
ファイル名： 03休暇規則.md
関連度スコア: 0.8649678930474555
テキスト：
3. 特別有給休暇

会社が特別に認めた有給の休暇です。

1. **リフレッシュ休暇**

   - **勤続5年**ごとに、連続した**5日間**のリフレッシュ休暇が取得できます。
   - リフレッシュ休暇は、有給休暇とは別に付与されます。

2. **ボ

◆実験2　（チャットエンジンはステートフル。★どこでそれがわかる？）

In [17]:
# 質問：2回目
response = chat_engine.chat("勤続年数が5年の場合は何日ですか？")

# 言語モデルからの回答を表示
print(response)

Added user message to memory: 勤続年数が5年の場合は何日ですか？
=== Calling Function ===
Calling function: query_engine_tool with args: {"input":"勤続年数が5年の場合の有給休暇の日数は何日ですか？"}
Got output: 勤続年数が5年の場合の有給休暇の日数は18日です。

勤続年数が5年の場合の有給休暇の日数は18日です。


◆引用もと表示

In [18]:
# 引用元を表示
for source in response.sources:
    for source_node in source.raw_output.source_nodes:
        print("ファイル名：", source_node.metadata["file_name"])
        print("関連度スコア:", source_node.score)
        print("テキスト：")
        print(source_node.node.text)
        print("-" * 50)  # 区切り線

ファイル名： 03休暇規則.md
関連度スコア: 0.8636387979410578
テキスト：
3. 特別有給休暇

会社が特別に認めた有給の休暇です。

1. **リフレッシュ休暇**

   - **勤続5年**ごとに、連続した**5日間**のリフレッシュ休暇が取得できます。
   - リフレッシュ休暇は、有給休暇とは別に付与されます。

2. **ボランティア休暇**

   - 社会貢献活動を支援するため、年間**2日間**のボランティア休暇を取得できます。
   - ボランティア休暇を取得する際は、活動内容を事前に上司へ報告してください。
--------------------------------------------------
ファイル名： 03休暇規則.md
関連度スコア: 0.8597444028938106
テキスト：
1. 年次有給休暇（有給休暇）

1. **有給休暇とは**

   - 有給休暇は、給与を受け取りながら休暇を取得できる制度です。
   - 心身のリフレッシュや私用のために自由に利用できます。

2. **付与日数**

   - 入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に、初めて有給休暇が付与されます。
   - 初年度は10日間の有給休暇が付与され、その後は勤続年数に応じて増加します。

     | 勤続年数 | 年次有給休暇日数 |
     |----------|----------------|
     | 0.5年    | 10日           |
     | 1.5年    | 11日           |
     | 2.5年    | 12日           |
     | 3.5年    | 14日           |
     | 4.5年    | 16日           |
     | 5.5年    | 18日           |
     | 6.5年以上 | 20日           |

3. **有給休暇の取得方法**

   - 有給休暇を取得する際は、原則として**3日前**までに上司に申請してください。
   - 緊急の場合は、当日の申請も可能ですが、できるだけ早めに連絡をお願いします。
   - 申請は、社内の休暇